In [1]:
# Cell 1 — parameters

LAT        = 16.8167
LON        = -2.9833
LEVEL      = 6
FROM_YEAR  = 1100
TO_YEAR    = 1200
HYBAS_ID   = 1060551560          # confirmed: both engines, both levels, same basin

BASE_URL   = 'http://localhost:8000'

In [2]:
# Cell 2 — imports and output path

import sys, json
from pathlib import Path
import requests

sys.path.insert(0, str(Path('.').resolve()))
import scripts.shared.db_utils as _dbu

ROOT = Path(_dbu.__file__).resolve().parents[2]
OUT  = ROOT / 'output' / 'edop' / 'areas'

In [3]:
# Cell 3 — fetch v0.3 signature and save to disk

url = (f'{BASE_URL}/api/signature'
       f'?lat={LAT}&lon={LON}'
       f'&bands=ABCDET&level={LEVEL}'
       f'&from_year={FROM_YEAR}&to_year={TO_YEAR}')

resp = requests.get(url)
resp.raise_for_status()
v3 = resp.json()

out_path = OUT / 'v03_timbuktu_signature.json'
out_path.write_text(json.dumps(v3, indent=2))

print(f'Status: {resp.status_code}')
print(f'Top-level keys: {list(v3.keys())}')
print(f'Bands present: {list(v3["profile_groups"].keys())}')
print(f'Basin id (v0.3 internal): {v3["id"]}')
print()

# Bands A–E use items list
for band in ['A', 'B', 'C', 'D', 'E']:
    grp   = v3['profile_groups'][band]
    items = grp.get('items', [])
    print(f'  Band {band}: {len(items)} items — {[i["key"] for i in items]}')

# Band T: LMR uses a single grid cell; HYDE aggregates n_cells within the basin
t    = v3['profile_groups']['T']
hyde = t.get('hyde_land_use') or {}
epochs = hyde.get('epochs') or []

print()
print(f'  Band T (LMR): grid_cell={t["grid_cell"]}  ({len(t.get("pdsi_series") or [])} annual values)')
print(f'    pdsi  mean={t["pdsi_mean"]}  min={t["pdsi_min"]}  max={t["pdsi_max"]}')
print(f'    temp  mean_anom_k={t["air_mean_anom_k"]}')
print(f'    precip mean_anom_mm_day={t["prate_mean_anom_mm_day"]}')
print(f'    volcanic_events={len(t.get("volcanic_events") or [])}')
print()
print(f'  Band T (HYDE): {len(epochs)} epoch snapshots (span endpoints only — not full series)')
for ep in epochs:
    print(f'    year={ep["year_ce"]}  n_cells={ep["n_cells"]}  basin_area={ep["basin_area_km2"]} km²')
    print(f'      cropland: {ep["cropland_km2"]} km² ({ep["cropland_pct"]}%)  '
          f'p10={ep["cropland_p10"]}  p90={ep["cropland_p90"]}  std={ep["cropland_std"]}')
    print(f'      grazing:  {ep["grazing_km2"]} km² ({ep["grazing_pct"]}%)  '
          f'p10={ep["grazing_p10"]}  p90={ep["grazing_p90"]}  std={ep["grazing_std"]}')

print()
print(f'Saved → {out_path}')

Status: 200
Top-level keys: ['id', 'eco_id', 'up_area', 'geom_geojson', 'elev_point', 'elev_source', 'elev_dataset', 'elev_resolution_m', 'relief_range_m', 'relief_position', 'profile_summary', 'profile_groups', 'meta']
Bands present: ['A', 'B', 'C', 'D', 'E', 'T']
Basin id (v0.3 internal): 1478

  Band A: 8 items — ['elev_min', 'elev_max', 'slope_avg', 'slope_upstream', 'stream_gradient', 'lith_class', 'karst', 'karst_upstream']
  Band B: 20 items — ['runoff', 'discharge_yr', 'discharge_min', 'discharge_max', 'river_area', 'river_area_upstream', 'gw_table_depth', 'pnv_majority', 'pnv_shares', 'pct_clay', 'pct_silt', 'pct_sand', 'pct_clay_upstream', 'pct_silt_upstream', 'pct_sand_upstream', 'wet_pct_grp1', 'wet_pct_grp2', 'wet_pct_grp1_upstream', 'wet_pct_grp2_upstream', 'wetland_class']
  Band C: 13 items — ['temp_yr', 'temp_min', 'temp_max', 'temp_yr_upstream', 'precip_yr', 'precip_yr_upstream', 'aridity', 'aridity_upstream', 'permafrost_extent', 'biome', 'ecoregion', 'freshwater_eco

In [4]:
# Cell 4 — resolver sign-off gate (run before full payload)
# Report the one-entry weighted set: hybas_id, area, weight, shortfall.
# Expected: hybas_id=1060551560, weight=1.0, shortfall=0.0

import sys, warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy', category=UserWarning)
sys.path.insert(0, str(Path('.').resolve()))

from scripts.shared.db_utils import db_connect
from scripts.edop.areas.engine import resolve_single_basin

conn = db_connect()
try:
    basin_set = resolve_single_basin(LAT, LON, LEVEL, conn)
finally:
    conn.close()

assert len(basin_set) == 1,        f'FAIL: expected 1 basin, got {len(basin_set)}'
assert basin_set['weight'].iloc[0] == 1.0, 'FAIL: weight must be 1.0'
assert int(basin_set['hybas_id'].iloc[0]) == HYBAS_ID, \
    f'FAIL: hybas_id={basin_set["hybas_id"].iloc[0]}, expected {HYBAS_ID}'

print('Resolver output:')
print(basin_set.to_string(index=False))
print()
print(f'shortfall = 0.0  (structural: query IS the basin)')
print(f'PASS — hybas_id={HYBAS_ID}, weight=1.0')

Resolver output:
  hybas_id  weight
1060551560     1.0

shortfall = 0.0  (structural: query IS the basin)
PASS — hybas_id=1060551560, weight=1.0


In [5]:
# Cell 5 — run v0.4 single_basin_signature and persist payload

from scripts.edop.areas.engine import single_basin_signature

conn = db_connect()
try:
    v4 = single_basin_signature(
        LAT, LON, conn,
        level=LEVEL,
        from_year=FROM_YEAR, to_year=TO_YEAR,
        include_detail=True,
    )
finally:
    conn.close()

out_path_v4 = OUT / 'v04_timbuktu_single_basin.json'
out_path_v4.write_text(json.dumps(v4, indent=2))

nb   = v4['neighborhood']
rows = v4['rows']

print(f'neighborhood: type={nb["type"]}  hybas_id={nb["hybas_id"]}  '
      f'n_units={nb["n_units"]}  level={nb["level"]}')
print(f'shortfall: {v4["shortfall"]}')
print(f'bands: {v4["bands"]}')
print(f'rows: {len(rows)}')
print()

from collections import Counter
by_method = Counter(r['method'] for r in rows)
for method, n in sorted(by_method.items()):
    print(f'  {method:<25} {n} rows')

t_rows = [r for r in rows if r['band'] == 'T']
if t_rows:
    print()
    print(f'  Band T: {len(t_rows)} rows')
    by_var = Counter(r['variable'] for r in t_rows)
    for var, n in sorted(by_var.items()):
        print(f'    {var:<20} {n}')

print()
print(f'Saved → {out_path_v4}')

neighborhood: type=basin  hybas_id=1060551560  n_units=1  level=6
shortfall: 0.0
bands: ['A', 'B', 'C', 'D', 'E', 'T']
rows: 373

  area_weighted             34 rows
  class_mixture             10 rows
  distribution_only         3 rows
  dominant_basin            3 rows
  extreme                   1 rows
  flag_fraction             1 rows
  global_forcing            10 rows
  grid_areal_collapsed      303 rows
  grid_areal_distribution   8 rows

  Band T: 321 rows
    evolv2k_vssi         10
    hyde_cropland        2
    hyde_grazing         2
    hyde_pasture         2
    hyde_rangeland       2
    lmr_air              101
    lmr_pdsi             101
    lmr_prate            101

Saved → /Users/karlg/Documents/repos/_edops/output/edop/areas/v04_timbuktu_single_basin.json


In [ ]:
# Cell 6 — bucket assignment: classify every v0.3 field and v0.4 row
# Buckets: 1=shared comparable, 2=v0.4-only, 3=v0.3-only explained, 4=transformed

# ── v0.3 field inventory ──────────────────────────────────────────────────────
v3_bands_items = {}
for band in ['A','B','C','D','E']:
    for item in v3['profile_groups'][band].get('items', []):
        v3_bands_items[item['key']] = item['value']

v3_top_level = {k: v for k, v in v3.items()
                if k not in ('profile_groups', 'geom_geojson', 'meta')}

# ── v0.4 row index ────────────────────────────────────────────────────────────
v4_by_var = {r['variable']: r for r in v4['rows'] if r['band'] != 'T'}

# ── bucket table (hard-coded classification) ──────────────────────────────────
# Each entry: (v3_key, v4_variable, bucket, note)
bucket_table = [
    # Bucket 1 — raw-vs-raw (dominant_basin / extreme)
    ('discharge_yr',               'discharge_yr',    1, 'raw-vs-raw'),
    ('discharge_max',              'discharge_max',   1, 'raw-vs-raw'),
    ('discharge_min',              'discharge_min',   1, 'raw-vs-raw'),
    ('river_area',                 'river_area',      1, 'raw-vs-raw'),
    # Bucket 1 — categorical raw-vs-raw (class_mixture)
    ('biome',                      'biome',           1, 'categorical'),
    ('zone_name',                  'zone_name',       1, 'categorical'),
    ('freshwater_ecoregion_name',  'freshwater_ecoregion_name', 1, 'categorical'),
    ('freshwater_ecoregion_class', 'freshwater_ecoregion_class', 1, 'categorical'),
    ('lith_class',                 'lith_class',      1, 'categorical'),
    ('pnv_majority',               'pnv_majority',    1, 'categorical'),
    ('wetland_class',              'wetland_class',   1, 'categorical'),
    ('land_cover_name',            'land_cover_name', 1, 'categorical'),
    # Bucket 1 — area_weighted (score-vs-rank)
    ('aridity',           'aridity',                  1, 'score-vs-rank'),
    ('aridity_upstream',  'aridity_upstream',         1, 'score-vs-rank'),
    ('cropland_extent',   'cropland_extent',          1, 'score-vs-rank'),
    ('cropland_extent_upstream', 'cropland_extent_upstream', 1, 'score-vs-rank'),
    ('dist_sink',         'dist_sink',                1, 'score-vs-rank'),
    ('elev_max',          'elev_max',                 1, 'score-vs-rank'),
    ('elev_min',          'elev_min',                 1, 'score-vs-rank'),
    ('gw_table_depth',    'gw_table_depth',           1, 'score-vs-rank'),
    ('human_footprint_09','human_footprint_09',       1, 'score-vs-rank'),
    ('human_footprint_09_upstream','human_footprint_09_upstream', 1, 'score-vs-rank'),
    ('karst',             'karst',                    1, 'score-vs-rank'),
    ('karst_upstream',    'karst_upstream',           1, 'score-vs-rank'),
    ('pasture_extent',    'pasture_extent',           1, 'score-vs-rank'),
    ('pasture_extent_upstream','pasture_extent_upstream', 1, 'score-vs-rank'),
    ('pct_clay',          'pct_clay',                 1, 'score-vs-rank'),
    ('pct_clay_upstream', 'pct_clay_upstream',        1, 'score-vs-rank'),
    ('pct_sand',          'pct_sand',                 1, 'score-vs-rank'),
    ('pct_sand_upstream', 'pct_sand_upstream',        1, 'score-vs-rank'),
    ('pct_silt',          'pct_silt',                 1, 'score-vs-rank'),
    ('pct_silt_upstream', 'pct_silt_upstream',        1, 'score-vs-rank'),
    ('permafrost_extent', 'permafrost_extent',        1, 'score-vs-rank'),
    ('pop_density',       'pop_density',              1, 'score-vs-rank'),
    ('precip_yr',         'precip_yr',                1, 'score-vs-rank'),
    ('precip_yr_upstream','precip_yr_upstream',       1, 'score-vs-rank'),
    ('runoff',            'runoff',                   1, 'score-vs-rank'),
    ('slope_avg',         'slope_avg',                1, 'score-vs-rank'),
    ('slope_upstream',    'slope_upstream',           1, 'score-vs-rank'),
    ('stream_gradient',   'stream_gradient',          1, 'score-vs-rank'),
    ('temp_yr',           'temp_yr',                  1, 'score-vs-rank'),
    ('temp_yr_upstream',  'temp_yr_upstream',         1, 'score-vs-rank'),
    ('wet_pct_grp1',      'wet_pct_grp1',             1, 'score-vs-rank'),
    ('wet_pct_grp1_upstream','wet_pct_grp1_upstream', 1, 'score-vs-rank'),
    ('wet_pct_grp2',      'wet_pct_grp2',             1, 'score-vs-rank'),
    ('wet_pct_grp2_upstream','wet_pct_grp2_upstream', 1, 'score-vs-rank'),
    # Bucket 1 — distribution_only (score-vs-rank)
    ('temp_max',          'temp_max',                 1, 'score-vs-rank'),
    ('temp_min',          'temp_min',                 1, 'score-vs-rank'),
    ('reservoir_vol',     'reservoir_vol',            1, 'score-vs-rank (distribution_only; upstream-only coalesce via rev_mc_usu)'),
    # Bucket 2 — v0.4 only (trust layer + derived)
    (None, 'coherence',          2, 'v0.4 trust layer'),
    (None, 'modality',           2, 'v0.4 trust layer'),
    (None, 'weight_at_zero',     2, 'v0.4 trust layer'),
    (None, 'score_suppressed',   2, 'v0.4 trust layer'),
    (None, 'caveat',             2, 'v0.4 trust layer'),
    (None, 'distribution',       2, 'v0.4 trust layer'),
    (None, 'representative_score (area_weighted)', 2, 'v0.4 scoring envelope'),
    # Bucket 3 — v0.3-only (each explained)
    ('id',               None, 3, 'internal DB row ID; v0.4 uses hybas_id in neighborhood'),
    ('up_area',          None, 3, 'BasinATLAS upstream area; not exposed in engine codebook'),
    ('geom_geojson',     None, 3, 'polygon geometry; not included in areal payload'),
    ('elev_point',       None, 3, 'point-geometry construct; areal engine has no point elevation'),
    ('elev_source',      None, 3, 'point-geometry construct'),
    ('elev_dataset',     None, 3, 'point-geometry construct'),
    ('elev_resolution_m',None, 3, 'point-geometry construct'),
    ('relief_range_m',   None, 3, 'point-profile construct; no analog in areal engine'),
    ('relief_position',  None, 3, 'point-profile construct'),
    ('profile_summary',  None, 3, 'point-profile summary; not in areal engine'),
    ('river_area_upstream', None, 3, 'in meta_df but deferred within B5 (engine.py line 928)'),
    ('gdp_avg',          None, 3, 'in _SKIP_API_KEYS (engine.py line 201)'),
    ('human_dev_idx',    None, 3, 'in _SKIP_API_KEYS (engine.py line 201)'),
    # Bucket 4 — same concept, different form
    ('endorheic',        'outlet_type',    4, 'raw int → derived class label (B4 synthesis)'),
    ('coast_flag',       'coast_fraction', 4, 'raw int → fraction (B4 synthesis)'),
    ('ecoregion',        'eco_id',         4, 'text label; v0.3 also has numeric eco_id top-level'),
    ('pnv_shares',       'pnv_majority.detail.mixture', 4, 'v0.3 dict → v0.4 detail sub-block'),
]

import pandas as pd
bdf = pd.DataFrame(bucket_table, columns=['v3_key','v4_variable','bucket','note'])
print(f'Total entries: {len(bdf)}')
print()
for b in [1,2,3,4]:
    sub = bdf[bdf['bucket']==b]
    print(f'Bucket {b}: {len(sub)} entries')
    print(sub[['v3_key','v4_variable','note']].to_string(index=False))
    print()

In [11]:
# Cell 7 — Bucket 1: raw-vs-raw checks (dominant_basin, extreme, class_mixture)
# area_weighted score-vs-rank validated by DB spot-check on 5 key variables.
# Spot-check SQL mirrors engine rank_expr exactly: zero-aware PARTITION BY when
# zero_fraction >= 0.20, plain PERCENT_RANK otherwise; NULLS LAST; -9999 guard.
# Engine result is authoritative; spot-check is the validation oracle.

TOL = 1e-3
ZF_THRESHOLD = 0.20

results_b1 = []

# ── raw-vs-raw: discharge + river_area ───────────────────────────────────────
for v3k, v4k in [('discharge_yr','discharge_yr'), ('discharge_max','discharge_max'),
                  ('discharge_min','discharge_min'), ('river_area','river_area')]:
    v3_val = v3_bands_items[v3k]
    v4_row = v4_by_var[v4k]
    v4_val = v4_row['representative_raw']
    match  = abs(v3_val - v4_val) < TOL
    verdict = 'match' if match else 'MISMATCH'
    results_b1.append({'variable': v3k, 'check': 'raw-vs-raw',
                        'v3': v3_val, 'v4': v4_val, 'verdict': verdict})
    print(f'{"PASS" if match else "FAIL"} {v3k}: v0.3={v3_val}  v0.4={v4_val}')

print()

# ── categorical raw-vs-raw (class_mixture) ────────────────────────────────────
v3_profile = {d['key']: d['value'] for d in v3['profile_summary']}

cat_pairs = [
    ('biome',                     v3_bands_items, 'biome'),
    ('freshwater_ecoregion_name', v3_bands_items, 'freshwater_ecoregion_name'),
    ('freshwater_ecoregion_class',v3_bands_items, 'freshwater_ecoregion_class'),
    ('lith_class',                v3_bands_items, 'lith_class'),
    ('pnv_majority',              v3_bands_items, 'pnv_majority'),
    ('wetland_class',             v3_bands_items, 'wetland_class'),
    ('zone_name',                 v3_profile,     'zone_name'),
    ('land_cover_name',           v3_profile,     'land_cover_name'),
]

for v3k, v3_src, v4k in cat_pairs:
    v3_val = v3_src.get(v3k)
    v4_val = v4_by_var.get(v4k, {}).get('representative_raw')
    match  = v3_val == v4_val
    verdict = 'match' if match else 'MISMATCH'
    results_b1.append({'variable': v3k, 'check': 'categorical',
                        'v3': v3_val, 'v4': v4_val, 'verdict': verdict})
    print(f'{"PASS" if match else "FAIL"} {v3k}: "{v4_val}"')

print()

# ── score-vs-rank spot-check (mirrors engine rank_expr) ──────────────────────
from scripts.shared.db_utils import db_connect
from scripts.edop.areas.engine import load_catalog

CODEBOOK = ROOT / 'documentation' / 'EDOPS_variable_catalog_v0.3.tsv'

def _spot_sql(db_col, method, zero_fraction, hybas_id):
    """SQL that mirrors engine rank_expr: zero-aware or plain, with -9999 guard."""
    nd  = f"({db_col} = -9999 OR {db_col} IS NULL)"
    val = (f"LN(1.0 + GREATEST(0.0, {db_col}::float))"
           if method == 'log_percentile' else f"{db_col}::float")
    if zero_fraction is not None and zero_fraction >= ZF_THRESHOLD:
        pos_val = f"CASE WHEN {nd} OR {db_col} <= 0 THEN NULL ELSE {val} END"
        score_expr = (
            f"CASE WHEN {nd} THEN NULL "
            f"WHEN {db_col} = 0 THEN 0.0 "
            f"ELSE PERCENT_RANK() OVER ("
            f"PARTITION BY CASE WHEN {db_col} > 0 THEN 1 ELSE 0 END "
            f"ORDER BY {pos_val} NULLS LAST) * 100 END"
        )
    else:
        order_expr = f"CASE WHEN {nd} THEN NULL ELSE {val} END"
        score_expr = (
            f"CASE WHEN {nd} THEN NULL "
            f"ELSE PERCENT_RANK() OVER (ORDER BY {order_expr} NULLS LAST) * 100 END"
        )
    return f"""
        SELECT score FROM (
            SELECT hybas_id, {score_expr} AS score
            FROM public.basin06
        ) ranked WHERE hybas_id = {hybas_id}
    """

conn = db_connect()
try:
    meta = load_catalog(6, CODEBOOK)
    spot_vars = ['aridity', 'temp_yr', 'precip_yr', 'dist_sink', 'pop_density']
    spot_meta = meta.loc[[v for v in spot_vars if v in meta.index]]

    for api_key, row in spot_meta.iterrows():
        zf       = row.get('zero_fraction')
        db_col   = row['db_col']
        pm       = row['position_method']
        sql      = _spot_sql(db_col, pm, zf, HYBAS_ID)
        db_rank  = float(conn.execute(sql).fetchone()[0])
        v4_score = v4_by_var[api_key]['representative_score']
        delta    = abs(db_rank - v4_score)
        match    = delta < 0.1
        verdict  = 'match' if match else 'MISMATCH'
        zf_flag  = f'  [zero_aware zf={zf:.2f}]' if (zf is not None and zf >= ZF_THRESHOLD) else ''
        results_b1.append({'variable': api_key, 'check': 'score-vs-rank (spot)',
                            'v3': v3_bands_items.get(api_key), 'v4': round(v4_score,2),
                            'v4_db_rank': round(db_rank,2), 'delta': round(delta,4),
                            'verdict': verdict})
        print(f'{"PASS" if match else "FAIL"} {api_key}: v0.4={v4_score:.2f}  db={db_rank:.2f}  Δ={delta:.4f}{zf_flag}')
finally:
    conn.close()

print()
b1_mismatches = [r for r in results_b1 if r['verdict'] == 'MISMATCH']
print(f'Bucket 1 summary: {len(results_b1)} checks, {len(b1_mismatches)} MISMATCH'
      + (' ← BUGS' if b1_mismatches else ' — all match'))

PASS discharge_yr: v0.3=499.327  v0.4=499.327
PASS discharge_max: v0.3=1049.821  v0.4=1049.821
PASS discharge_min: v0.3=277.059  v0.4=277.059
PASS river_area: v0.3=2802.396  v0.4=2802.396

PASS biome: "Flooded Grasslands & Savannas"
PASS freshwater_ecoregion_name: "Inner Niger Delta"
PASS freshwater_ecoregion_class: "Tropical and subtropical floodplain rivers and wetlands"
PASS lith_class: "Unconsolidated Sediments (SU)"
PASS pnv_majority: "Open shrubland"
PASS wetland_class: "Freshwater marsh, floodplain"
PASS zone_name: "Extremely hot and xeric"
PASS land_cover_name: "Sparse herbaceous or sparse shrub cover"

PASS aridity: v0.4=10.82  db=10.82  Δ=0.0003
PASS temp_yr: v0.4=98.07  db=98.07  Δ=0.0034
PASS precip_yr: v0.4=16.89  db=16.89  Δ=0.0044
PASS dist_sink: v0.4=80.22  db=80.22  Δ=0.0040  [zero_aware zf=0.33]
PASS pop_density: v0.4=71.41  db=71.41  Δ=0.0023

Bucket 1 summary: 17 checks, 0 MISMATCH — all match


In [12]:
# Cell 8 — Bucket 3: v0.3-only inventory; each absence must be explained
# Expected: every row maps to a known reason; any UNEXPLAINED row is a bug.

b3_rows = bdf[bdf['bucket'] == 3]
print(f'Bucket 3 — {len(b3_rows)} v0.3-only fields:\n')

EXPLAIN_CATEGORIES = {
    'id':               'expected: internal DB row ID; v0.4 uses hybas_id in neighborhood',
    'up_area':          'expected: BasinATLAS upstream area; not exposed in engine codebook',
    'geom_geojson':     'expected: polygon geometry; not included in areal payload rows',
    'elev_point':       'expected: point-elevation lookup; areal engine has no point query',
    'elev_source':      'expected: point-elevation metadata',
    'elev_dataset':     'expected: point-elevation metadata',
    'elev_resolution_m':'expected: point-elevation metadata',
    'relief_range_m':   'expected: point-profile construct (elev_max − elev_min at point)',
    'relief_position':  'expected: point-profile construct',
    'profile_summary':  'expected: point-profile summary block; no areal analog',
    'river_area_upstream':'expected: deferred within B5 (engine.py, register)',
    'gdp_avg':          'expected: in _SKIP_API_KEYS (engine.py line 201)',
    'human_dev_idx':    'expected: in _SKIP_API_KEYS (engine.py line 201)',
}

for _, row in b3_rows.iterrows():
    v3k = row['v3_key']
    expl = EXPLAIN_CATEGORIES.get(v3k, 'UNEXPLAINED')
    status = 'OK ' if expl.startswith('expected') else 'BUG'
    print(f'  {status}  {v3k:<22}  {expl}')

unexplained = [v for v in b3_rows['v3_key'] if EXPLAIN_CATEGORIES.get(v, 'UNEXPLAINED') == 'UNEXPLAINED']
print()
print(f'Bucket 3 summary: {len(b3_rows)} absences, {len(unexplained)} UNEXPLAINED'
      + (' ← BUGS' if unexplained else ' — all explained'))

Bucket 3 — 13 v0.3-only fields:

  OK   id                      expected: internal DB row ID; v0.4 uses hybas_id in neighborhood
  OK   up_area                 expected: BasinATLAS upstream area; not exposed in engine codebook
  OK   geom_geojson            expected: polygon geometry; not included in areal payload rows
  OK   elev_point              expected: point-elevation lookup; areal engine has no point query
  OK   elev_source             expected: point-elevation metadata
  OK   elev_dataset            expected: point-elevation metadata
  OK   elev_resolution_m       expected: point-elevation metadata
  OK   relief_range_m          expected: point-profile construct (elev_max − elev_min at point)
  OK   relief_position         expected: point-profile construct
  OK   profile_summary         expected: point-profile summary block; no areal analog
  OK   river_area_upstream     expected: deferred within B5 (engine.py, register)
  OK   gdp_avg                 expected: in _SKIP_API_K

In [15]:
# Cell 9 — Bucket 4: transformation checks
# (a) outlet_type synthesis: endorheic+coast_flag → outlet_type class label
# (b) eco_id: v0.3 numeric + text label vs v0.4 class_mixture row (text label)
# (c) pnv_shares: v0.3 intra-basin grid distribution vs v0.4 cross-basin mixture

results_b4 = []

# ── (a) outlet_type synthesis ─────────────────────────────────────────────────
endorheic_v3  = v3_bands_items['endorheic']
coast_flag_v3 = v3_bands_items['coast_flag']

outlet_v4     = v4_by_var['outlet_type']['representative_raw']
coast_frac_v4 = v4_by_var['coast_fraction']['representative_raw']

expected_outlet = ('Exorheic, coastal'      if coast_flag_v3 == 1 else
                   'Endorheic, coastal'     if endorheic_v3 == 1 and coast_flag_v3 == 1 else
                   'Endorheic, non-coastal' if endorheic_v3 == 1 else
                   'Exorheic, non-coastal')
expected_coast_frac = float(coast_flag_v3)

ok_outlet = (outlet_v4 == expected_outlet)
ok_coast  = abs(coast_frac_v4 - expected_coast_frac) < 1e-6

print(f'outlet_type synthesis:')
print(f'  v0.3: endorheic={endorheic_v3}, coast_flag={coast_flag_v3}')
print(f'  expected: "{expected_outlet}"  v0.4: "{outlet_v4}"')
print(f'  {"PASS" if ok_outlet else "MISMATCH"}')
print()
print(f'coast_fraction synthesis:')
print(f'  expected: {expected_coast_frac}  v0.4: {coast_frac_v4}')
print(f'  {"PASS" if ok_coast else "MISMATCH"}')
print()

results_b4.append({'variable': 'outlet_type',    'verdict': 'match' if ok_outlet else 'MISMATCH'})
results_b4.append({'variable': 'coast_fraction', 'verdict': 'match' if ok_coast  else 'MISMATCH'})

# ── (b) eco_id transformation ─────────────────────────────────────────────────
eco_id_v3_num  = v3['eco_id']
eco_id_v3_text = v3_bands_items['ecoregion']
eco_id_v4_text = v4_by_var['eco_id']['representative_raw']

ok_eco = (eco_id_v3_text == eco_id_v4_text)
print(f'eco_id transformation:')
print(f'  v0.3 numeric: {eco_id_v3_num}  v0.3 text: "{eco_id_v3_text}"')
print(f'  v0.4 class_mixture label: "{eco_id_v4_text}"')
print(f'  {"PASS" if ok_eco else "MISMATCH"} (text labels match; numeric ID not preserved — expected)')
print()
results_b4.append({'variable': 'eco_id', 'verdict': 'explained-difference' if ok_eco else 'MISMATCH'})

# ── (c) pnv_shares — within-basin vs cross-basin distribution ─────────────────
# v0.3: within-basin grid-cell PNV distribution via bespoke pnv_pc_* column reads.
# v0.4: cross-basin class_mixture of per-basin majority labels — correct for multi-
#   basin areal; loses within-basin distribution at n=1 (100% one class).
#   Modal class must match; full-distribution difference is architectural, not a bug.
#   Deferred: see register "pnv_shares within-basin distribution" + "multi-column
#   variable gap."
pnv_modal_v3  = v3_bands_items['pnv_majority']
pnv_shares_v3 = v3_bands_items['pnv_shares']
pnv_modal_v4  = v4_by_var['pnv_majority']['representative_raw']
pnv_detail_v4 = v4_by_var['pnv_majority'].get('detail', {}).get('mixture', [])
pnv_v4_dict   = {m['class_label']: round(m['weight'] * 100) for m in pnv_detail_v4}

modal_match = (pnv_modal_v3 == pnv_modal_v4)
print(f'pnv_shares transformation:')
print(f'  modal class: v0.3="{pnv_modal_v3}"  v0.4="{pnv_modal_v4}"  → {"PASS" if modal_match else "MISMATCH"}')
print(f'  v0.3 intra-basin distribution: {pnv_shares_v3}')
print(f'  v0.4 cross-basin mixture (n=1): {pnv_v4_dict}')
print(f'  explained-difference: v0.4 cross-basin mixture loses within-basin grain at n=1')
print(f'    (deferred: register "pnv_shares within-basin distribution")')
results_b4.append({'variable': 'pnv_shares',
                   'verdict': 'explained-difference' if modal_match else 'MISMATCH'})

print()
b4_mismatches = [r for r in results_b4 if r['verdict'] == 'MISMATCH']
print(f'Bucket 4 summary: {len(results_b4)} checks, {len(b4_mismatches)} MISMATCH'
      + (' ← BUGS' if b4_mismatches else ' — all verified (1 explained-difference: pnv_shares)'))

outlet_type synthesis:
  v0.3: endorheic=0, coast_flag=0
  expected: "Exorheic, non-coastal"  v0.4: "Exorheic, non-coastal"
  PASS

coast_fraction synthesis:
  expected: 0.0  v0.4: 0.0
  PASS

eco_id transformation:
  v0.3 numeric: 71  v0.3 text: "Inner Niger Delta flooded savanna"
  v0.4 class_mixture label: "Inner Niger Delta flooded savanna"
  PASS (text labels match; numeric ID not preserved — expected)

pnv_shares transformation:
  modal class: v0.3="Open shrubland"  v0.4="Open shrubland"  → PASS
  v0.3 intra-basin distribution: {'Desert': 16, 'Open shrubland': 51, 'Grassland/steppe': 33}
  v0.4 cross-basin mixture (n=1): {'Open shrubland': 100}
  explained-difference: v0.4 cross-basin mixture loses within-basin grain at n=1
    (deferred: register "pnv_shares within-basin distribution")

Bucket 4 summary: 4 checks, 0 MISMATCH — all verified (1 explained-difference: pnv_shares)


In [17]:
# Cell 10 — assemble and save comparison TSV

rows_tsv = []

# Bucket 1 — raw-vs-raw and categorical
for r in results_b1:
    rows_tsv.append({
        'bucket':    1,
        'variable':  r['variable'],
        'check':     r['check'],
        'v3_value':  r.get('v3', ''),
        'v4_value':  r.get('v4', r.get('v4_db_rank', '')),
        'delta':     r.get('delta', ''),
        'verdict':   r['verdict'],
        'note':      '',
    })

# Bucket 2 — v0.4-only (no values to compare; noted as expected)
for _, row in bdf[bdf['bucket'] == 2].iterrows():
    rows_tsv.append({
        'bucket': 2, 'variable': row['v4_variable'], 'check': 'v0.4-only',
        'v3_value': '', 'v4_value': '', 'delta': '',
        'verdict': 'expected', 'note': row['note'],
    })

# Bucket 3 — v0.3-only (each explained)
for _, row in bdf[bdf['bucket'] == 3].iterrows():
    expl = EXPLAIN_CATEGORIES.get(row['v3_key'], 'UNEXPLAINED')
    rows_tsv.append({
        'bucket': 3, 'variable': row['v3_key'], 'check': 'v0.3-only',
        'v3_value': v3_top_level.get(row['v3_key'], v3_bands_items.get(row['v3_key'], '')),
        'v4_value': '', 'delta': '',
        'verdict': 'explained' if expl.startswith('expected') else 'UNEXPLAINED',
        'note': expl,
    })

# Bucket 4 — transformations; match bdf on v4_variable OR v3_key
b4_bdf = bdf[bdf['bucket'] == 4]
for r in results_b4:
    var = r['variable']
    match = b4_bdf[
        b4_bdf['v4_variable'].str.startswith(var) | (b4_bdf['v3_key'] == var)
    ]
    note = match.iloc[0]['note'] if len(match) else ''
    rows_tsv.append({
        'bucket': 4, 'variable': var, 'check': 'transformation',
        'v3_value': '', 'v4_value': '', 'delta': '',
        'verdict': r['verdict'], 'note': note,
    })

comparison_df = pd.DataFrame(rows_tsv, columns=[
    'bucket', 'variable', 'check', 'v3_value', 'v4_value', 'delta', 'verdict', 'note'
])

out_tsv = OUT / 'wo14_comparison.tsv'
comparison_df.to_csv(out_tsv, sep='\t', index=False)

print(f'Comparison TSV: {len(comparison_df)} rows saved → {out_tsv}')
print()
print('Verdict counts:')
print(comparison_df['verdict'].value_counts().to_string())
print()
bugs = comparison_df[comparison_df['verdict'].isin(['MISMATCH','UNEXPLAINED'])]
if len(bugs):
    print(f'BUGS ({len(bugs)}):')
    print(bugs[['bucket','variable','verdict','note']].to_string(index=False))
else:
    print('No bugs — all bucket 1 match, all bucket 3 explained, all bucket 4 verified.')

Comparison TSV: 41 rows saved → /Users/karlg/Documents/repos/_edops/output/edop/areas/wo14_comparison.tsv

Verdict counts:
verdict
match                   19
explained               13
expected                 7
explained-difference     2

No bugs — all bucket 1 match, all bucket 3 explained, all bucket 4 verified.


In [18]:
# Cell 11 — Part 3: n=1 degeneracy assertions (Bands A–E only)
# At n=1 the aggregator should degrade cleanly. Failures here are engine bugs.

basin_rows = [r for r in v4['rows'] if r['band'] != 'T']

failures = []

def chk(label, ok, detail=''):
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        failures.append(f'{label}: {detail}')
    print(f'  {status}  {label}' + (f'  [{detail}]' if detail else ''))

print(f'Basin rows: {len(basin_rows)}')
print()

# ── 1. coherence = concentrated on every row that carries it ─────────────────
print('1. coherence = concentrated where set:')
for r in basin_rows:
    coh = r.get('coherence')
    if coh is not None:
        chk(f'{r["variable"]} ({r["method"]})', coh == 'concentrated',
            f'got {coh!r}')
print()

# ── 2. modality never two_regime ──────────────────────────────────────────────
print('2. modality ≠ two_regime where set:')
for r in basin_rows:
    mod = r.get('modality')
    if mod is not None:
        chk(f'{r["variable"]} ({r["method"]})', mod != 'two_regime',
            f'got {mod!r}')
print()

# ── 3. score_suppressed never True ────────────────────────────────────────────
print('3. score_suppressed never True:')
bad = [r['variable'] for r in basin_rows if r.get('score_suppressed')]
chk('all rows', len(bad) == 0, f'suppressed: {bad}' if bad else '')
print()

# ── 4. weight_at_zero ∈ {0.0, 1.0} where set ─────────────────────────────────
print('4. weight_at_zero ∈ {0.0, 1.0} where set:')
for r in basin_rows:
    waz = r.get('weight_at_zero')
    if waz is not None:
        chk(f'{r["variable"]}', waz in (0.0, 1.0), f'got {waz}')
print()

# ── 5. coverage = 1.0 for all ok rows ─────────────────────────────────────────
print('5. coverage = 1.0 for all ok rows:')
bad = [r['variable'] for r in basin_rows
       if r.get('status') == 'ok' and abs((r.get('coverage') or 0) - 1.0) > 1e-6]
chk('all ok rows', len(bad) == 0, f'coverage < 1: {bad}' if bad else '')
print()

# ── 6. class_mixture: 100% one class, modal_share = 1.0, n_classes = 1 ───────
print('6. class_mixture: single class, modal_share = 1.0:')
for r in basin_rows:
    if r['method'] != 'class_mixture':
        continue
    detail = r.get('detail') or {}
    modal_share = detail.get('modal_share')
    n_classes   = detail.get('n_classes')
    mixture     = detail.get('mixture', [])
    chk(f'{r["variable"]} modal_share=1.0',
        modal_share is not None and abs(modal_share - 1.0) < 1e-4,
        f'got {modal_share}')
    chk(f'{r["variable"]} n_classes=1',
        n_classes == 1,
        f'got {n_classes}')
    chk(f'{r["variable"]} mixture has 1 entry',
        len(mixture) == 1,
        f'got {len(mixture)} entries')
print()

print('─' * 60)
if failures:
    print(f'FAIL — {len(failures)} assertion(s) failed:')
    for f in failures:
        print(f'  • {f}')
else:
    print(f'PASS — all degeneracy assertions hold at n=1')

Basin rows: 52

1. coherence = concentrated where set:
  PASS  aridity (area_weighted)  [got 'concentrated']
  PASS  aridity_upstream (area_weighted)  [got 'concentrated']
  PASS  cropland_extent_upstream (area_weighted)  [got 'concentrated']
  PASS  dist_sink (area_weighted)  [got 'concentrated']
  PASS  elev_max (area_weighted)  [got 'concentrated']
  PASS  elev_min (area_weighted)  [got 'concentrated']
  PASS  gw_table_depth (area_weighted)  [got 'concentrated']
  PASS  human_footprint_09 (area_weighted)  [got 'concentrated']
  PASS  human_footprint_09_upstream (area_weighted)  [got 'concentrated']
  PASS  pasture_extent (area_weighted)  [got 'concentrated']
  PASS  pasture_extent_upstream (area_weighted)  [got 'concentrated']
  PASS  pct_clay (area_weighted)  [got 'concentrated']
  PASS  pct_clay_upstream (area_weighted)  [got 'concentrated']
  PASS  pct_sand (area_weighted)  [got 'concentrated']
  PASS  pct_sand_upstream (area_weighted)  [got 'concentrated']
  PASS  pct_silt (area

In [6]:
# Cell 12 — Part 4: Band T reference checks
# LMR: series stats vs v0.3 single-cell reference
# HYDE: n_units, per-cell means, distribution stats vs v0.3 epoch-for-epoch
# Volcanic: count vs v0.3 volcanic_events=4
# WO15 section at bottom: diff post-fix values against WO14 baseline; check w_eff

import statistics

t_rows = [r for r in v4['rows'] if r['band'] == 'T']

# ── v0.3 reference values (from cell 3 output) ───────────────────────────────
V3_LMR = {
    'pdsi': {'mean': -0.0393, 'min': -0.3804, 'max': 0.2768},
    'air':  {'mean': -0.1246},
    'prate':{'mean': -0.0153},
    'grid_cell': {'lat': 16.0, 'lon': -2.0},
}
V3_HYDE = {
    1100: {'n_cells': 45, 'basin_area': 3687.8,
           'cropland_km2': 1.932, 'cropland_pct': 0.05, 'cropland_p10': 0.0,  'cropland_p90': 0.08,  'cropland_std': 0.032,
           'grazing_km2':185.728, 'grazing_pct':  5.04, 'grazing_p10':  0.218, 'grazing_p90':  8.631, 'grazing_std':  6.804},
    1200: {'n_cells': 45, 'basin_area': 3687.8,
           'cropland_km2': 2.067, 'cropland_pct': 0.06, 'cropland_p10': 0.0,  'cropland_p90': 0.086, 'cropland_std': 0.035,
           'grazing_km2':198.399, 'grazing_pct':  5.38, 'grazing_p10':  0.208, 'grazing_p90':  9.244, 'grazing_std':  7.307},
}
V3_VOLCANIC_EVENTS = 4

# ── WO14 pre-WO15 baseline (from saved notebook outputs before engine edit) ───
# Used in the WO15 validation section below to diff before vs after.
V4_WO14 = {
    'lmr': {
        'pdsi':  {'mean': -0.0404, 'min': -0.3776, 'max': 0.2712},
        'air':   {'mean': -0.1252},
        'prate': {'mean': -0.0151},
    },
    'hyde': {
        1100: {
            'cropland': {'mean': 0.0414, 'p10': 0.0000, 'p90': 0.0801, 'sd': 0.0321, 'n_units': 80},
            'grazing':  {'mean': 4.4369, 'p10': 0.2183, 'p90': 8.7741, 'sd': 7.1450, 'n_units': 80},
        },
        1200: {
            'cropland': {'mean': 0.0443, 'p10': 0.0000, 'p90': 0.0857, 'sd': 0.0344, 'n_units': 80},
            'grazing':  {'mean': 4.7409, 'p10': 0.2079, 'p90': 9.3998, 'sd': 7.6734, 'n_units': 80},
        },
    },
}

TOL_LMR  = 0.01   # acceptable Δ given 3-cell collapse vs v0.3 single-cell
TOL_DIST = 0.05   # acceptable Δ for distribution stats (p10/p90/sd in km²/cell)

print('═' * 60)
print('LMR — sub-resolution collapse check')
print('═' * 60)

for v4_var, clim_var, v3ref in [
    ('lmr_pdsi',  'pdsi',  V3_LMR['pdsi']),
    ('lmr_air',   'air',   V3_LMR['air']),
    ('lmr_prate', 'prate', V3_LMR['prate']),
]:
    rows = [r for r in t_rows if r['variable'] == v4_var]
    vals = [r['representative_raw'] for r in rows]
    distrib = rows[0].get('distribution')
    n_units = rows[0].get('n_units')
    mean_v4 = statistics.mean(vals)
    print(f'\n{v4_var} (n={len(vals)}, n_units={n_units}, distribution={distrib!r}):')
    print(f'  mean: v0.3={v3ref["mean"]:+.4f}  v0.4={mean_v4:+.4f}  Δ={abs(mean_v4-v3ref["mean"]):.4f}  '
          f'{"ok" if abs(mean_v4-v3ref["mean"]) < TOL_LMR else "NOTE"}')
    if 'min' in v3ref:
        min_v4 = min(vals); max_v4 = max(vals)
        print(f'  min:  v0.3={v3ref["min"]:+.4f}  v0.4={min_v4:+.4f}  Δ={abs(min_v4-v3ref["min"]):.4f}  '
              f'{"ok" if abs(min_v4-v3ref["min"]) < TOL_LMR else "NOTE"}')
        print(f'  max:  v0.3={v3ref["max"]:+.4f}  v0.4={max_v4:+.4f}  Δ={abs(max_v4-v3ref["max"]):.4f}  '
              f'{"ok" if abs(max_v4-v3ref["max"]) < TOL_LMR else "NOTE"}')

print(f'\nNote: n_units=3 means basin polygon clips 3 LMR cells; ECC collapses to')
print(f'area-weighted mean, not purely the (16.0, -2.0) cell v0.3 used. Small Δ expected.')

print()
print('═' * 60)
print('HYDE — epoch-for-epoch check')
print('═' * 60)

for epoch in [1100, 1200]:
    ref = V3_HYDE[epoch]
    print(f'\nEpoch {epoch}:')
    for v4_var, label, ref_mean_key, ref_p10, ref_p90, ref_sd in [
        ('hyde_cropland', 'cropland',
         'cropland_km2', 'cropland_p10', 'cropland_p90', 'cropland_std'),
        ('hyde_grazing',  'grazing',
         'grazing_km2',  'grazing_p10',  'grazing_p90',  'grazing_std'),
    ]:
        r = next(x for x in t_rows if x['variable']==v4_var and x['year']==epoch)
        d = r.get('detail') or {}
        n4 = r['n_units']
        mean4 = r['representative_raw']
        mean3 = ref[ref_mean_key] / ref['n_cells']
        p10_3, p90_3, sd_3 = ref[ref_p10], ref[ref_p90], ref[ref_sd]
        p10_4, p90_4, sd_4 = d.get('p10',0), d.get('p90',0), d.get('sd',0)
        w_eff = d.get('w_eff', 'MISSING')
        print(f'  {label}: n_units v0.3={ref["n_cells"]} v0.4={n4}  '
              f'{"NOTE: n_units mismatch" if n4 != ref["n_cells"] else "n_units match"}  '
              f'w_eff={w_eff}')
        print(f'    mean/cell: v0.3={mean3:.4f}  v0.4={mean4:.4f}  Δ={abs(mean4-mean3):.4f}')
        print(f'    p10:  v0.3={p10_3:.4f}  v0.4={p10_4:.4f}  Δ={abs(p10_4-p10_3):.4f}')
        print(f'    p90:  v0.3={p90_3:.4f}  v0.4={p90_4:.4f}  Δ={abs(p90_4-p90_3):.4f}')
        print(f'    sd:   v0.3={sd_3:.4f}  v0.4={sd_4:.4f}  Δ={abs(sd_4-sd_3):.4f}')

print(f'\nNote: n_units=80 vs v0.3 n_cells=45. v0.3 used centroid-in-polygon filter;')
print(f'v0.4 uses ST_Intersects. The 35 extra boundary cells are included at their')
print(f'fractional coverage weight (WO15). n_units mismatch is structural, not a bug.')

print()
print('═' * 60)
print('Volcanic — global_forcing count check')
print('═' * 60)

volc = sorted([r for r in t_rows if r['variable']=='evolv2k_vssi'], key=lambda r: r['year'])
n_v4 = len(volc)
large = [r for r in volc if r['representative_raw'] > 5.0]
print(f'\nv0.3 volcanic_events = {V3_VOLCANIC_EVENTS}')
print(f'v0.4 rows returned  = {n_v4}')
print(f'v0.4 rows with vssi > 5 Tg S: {len(large)} (years: {[r["year"] for r in large]})')
print(f'\nAll v0.4 volcanic rows:')
for r in volc:
    flag = ' ← vssi > 5' if r['representative_raw'] > 5.0 else ''
    print(f'  year={r["year"]}  vssi={r["representative_raw"]}{flag}')
print(f'\nNote: v0.3 filtered by VSSI threshold (~5 Tg S) → 4 notable events.')
print(f'v0.4 returns all eVolv2k rows in span (no threshold). Not a bug; a display/filter')
print(f'design difference. The 4 large events are present in v0.4 (years 1108/1171/1182/1191).')

print()
print('═' * 60)
print('WO15 validation — before vs after fractional-overlap weighting')
print('═' * 60)
print('Expected at this fixture (16°N, equal-size HYDE cells): tiny shifts, w_eff present.')
print('LMR: 3 cells at same latitude → equal areas → weights identical → zero change.')
print()

wo15_failures = []

# LMR: should be identical (equal-size cells, same weights)
print('LMR shift (should be ~0):')
for v4_var, clim_var in [('lmr_pdsi','pdsi'), ('lmr_air','air'), ('lmr_prate','prate')]:
    rows = [r for r in t_rows if r['variable'] == v4_var]
    mean_new = statistics.mean(r['representative_raw'] for r in rows)
    mean_old = V4_WO14['lmr'][clim_var]['mean']
    delta = abs(mean_new - mean_old)
    flag = 'INVESTIGATE' if delta > 0.001 else 'ok'
    if flag != 'ok':
        wo15_failures.append(f'LMR {clim_var} mean shifted {delta:.6f} — unexpected at equal-size cells')
    print(f'  {v4_var}: old={mean_old:+.4f}  new={mean_new:+.4f}  Δ={delta:.6f}  {flag}')

print()
print('HYDE shift (small expected; w_eff confirms fix ran):')
for epoch in [1100, 1200]:
    ref_old = V4_WO14['hyde'][epoch]
    print(f'  Epoch {epoch}:')
    for v4_var, label in [('hyde_cropland','cropland'), ('hyde_grazing','grazing')]:
        r = next(x for x in t_rows if x['variable']==v4_var and x['year']==epoch)
        d = r.get('detail') or {}
        w_eff = d.get('w_eff')
        mean_new, p10_new, p90_new, sd_new = (
            r['representative_raw'], d.get('p10',0), d.get('p90',0), d.get('sd',0))
        old = ref_old[label]
        w_eff_ok = isinstance(w_eff, (int, float))
        if not w_eff_ok:
            wo15_failures.append(f'HYDE {label} epoch {epoch}: w_eff missing — WO15 fix did not run')
        print(f'    {label}: w_eff={w_eff}  {"PASS" if w_eff_ok else "FAIL — w_eff missing"}')
        for stat, new_v, old_v in [
            ('mean', mean_new, old['mean']),
            ('p10',  p10_new,  old['p10']),
            ('p90',  p90_new,  old['p90']),
            ('sd',   sd_new,   old['sd']),
        ]:
            delta = abs(new_v - old_v)
            # > 5% relative shift would be surprising at equal-size cells
            rel = delta / old_v if old_v != 0 else 0
            flag = 'INVESTIGATE' if rel > 0.05 else 'ok'
            if flag != 'ok':
                wo15_failures.append(f'HYDE {label} epoch {epoch} {stat}: {rel:.1%} shift — investigate')
            print(f'      {stat}: old={old_v:.4f}  new={new_v:.4f}  Δ={delta:.4f}  ({rel:.3%})  {flag}')

print()
if wo15_failures:
    print(f'WO15 ISSUES ({len(wo15_failures)}):')
    for f in wo15_failures:
        print(f'  • {f}')
else:
    print('WO15 PASS — w_eff present, shifts within expected range for equal-size cells.')

════════════════════════════════════════════════════════════
LMR — sub-resolution collapse check
════════════════════════════════════════════════════════════

lmr_pdsi (n=101, n_units=3, distribution='collapsed_subresolution'):
  mean: v0.3=-0.0393  v0.4=-0.0404  Δ=0.0011  ok
  min:  v0.3=-0.3804  v0.4=-0.3776  Δ=0.0028  ok
  max:  v0.3=+0.2768  v0.4=+0.2711  Δ=0.0057  ok

lmr_air (n=101, n_units=3, distribution='collapsed_subresolution'):
  mean: v0.3=-0.1246  v0.4=-0.1252  Δ=0.0006  ok

lmr_prate (n=101, n_units=3, distribution='collapsed_subresolution'):
  mean: v0.3=-0.0153  v0.4=-0.0151  Δ=0.0002  ok

Note: n_units=3 means basin polygon clips 3 LMR cells; ECC collapses to
area-weighted mean, not purely the (16.0, -2.0) cell v0.3 used. Small Δ expected.

════════════════════════════════════════════════════════════
HYDE — epoch-for-epoch check
════════════════════════════════════════════════════════════

Epoch 1100:
  cropland: n_units v0.3=45 v0.4=80  NOTE: n_units mismatch  w_eff=